# Week 67, Day 4: Async pitfalls

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w067d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def finish_times(tasks):
    "Each step is ('wait', t): other tasks may run; or ('block', t): nothing else can run."
    clock, finished, running = 0, {}, {name: list(steps) for name, steps in tasks}
    waits = {}
    while running:
        for name in list(running):
            if waits.get(name, 0) > clock:
                continue
            if not running[name]:
                finished[name] = max(clock, waits.get(name, 0))
                del running[name]
                continue
            kind, t = running[name].pop(0)
            if kind == "block":
                clock += t                 # the whole loop is stuck
            else:
                waits[name] = clock + t
        if running and all(waits.get(n, 0) > clock for n in running):
            clock = min(waits[n] for n in running)
    return finished

polite = [("a", [("wait", 5)]), ("b", [("wait", 1)])]
rude = [("a", [("block", 5)]), ("b", [("wait", 1)])]
print(finish_times(polite), finish_times(rude))

In [ ]:
def latency_of_others(blocking_ms, others):
    "How long each other task waits before it can even start, if one task blocks first."
    return [blocking_ms] * others

print(latency_of_others(200, 3), "ms: every request waits for the blocking one")

def with_offload(blocking_ms, others, thread_start_ms=1):
    return [thread_start_ms] * others            # the loop is free while a thread does the work

print(with_offload(200, 3), "ms with asyncio.to_thread")

In [ ]:
import warnings

async def is_allowed(user):
    return user != "mallory"

def check_wrongly(user):
    result = is_allowed(user)          # forgot await: a coroutine object
    allowed = bool(result)
    result.close()                     # tidy up the never-run coroutine
    return allowed

print(check_wrongly("mallory"), "(a coroutine object is always true)")

async def check_rightly(user):
    return await is_allowed(user)

c = check_rightly("mallory")
try:
    c.send(None)
except StopIteration as done:
    print(done.value)

In [ ]:
from concurrent.futures import Future

background = set()

def track(task):
    background.add(task)
    task.add_done_callback(background.discard)
    task.add_done_callback(report_errors)
    return task

errors = []
def report_errors(task):
    if not task.cancelled() and task.exception() is not None:
        errors.append(repr(task.exception()))

a, b = track(Future()), track(Future())
print(len(background))
a.set_result("ok")
b.set_exception(ValueError("disk full"))
print(len(background), errors)

In [ ]:
import asyncio

print(issubclass(asyncio.CancelledError, Exception), issubclass(asyncio.CancelledError, BaseException))

class Pause:
    def __await__(self):
        yield

async def well_behaved(log):
    try:
        while True:
            await Pause()
    finally:
        log.append("cleaned up")

async def swallows(log):
    while True:
        try:
            await Pause()
        except asyncio.CancelledError:
            log.append("ignored a cancellation")

for coro_fn in (well_behaved, swallows):
    log = []
    c = coro_fn(log)
    c.send(None)
    try:
        c.throw(asyncio.CancelledError())
        outcome = "still running"
    except asyncio.CancelledError:
        outcome = "cancelled"
    print(coro_fn.__name__, outcome, log)
    try:
        c.close()
    except RuntimeError:
        print("  and it even ignored close()")

In [ ]:
def book_all(requests, seats, use_lock):
    "Every request checks, awaits a save, then marks the seat taken."
    taken, booked, checked = set(), [], []
    if use_lock:
        for who, seat in requests:              # with a lock, each check-save-mark runs as one unit
            if seat not in taken:
                taken.add(seat)
                booked.append((who, seat))
        return booked
    for who, seat in requests:                  # all checks happen before any save finishes
        checked.append((who, seat, seat not in taken))
    for who, seat, free in checked:
        if free:
            taken.add(seat)
            booked.append((who, seat))
    return booked

requests = [("ada", "12A"), ("ben", "12A")]
print(book_all(requests, ["12A"], use_lock=False))
print(book_all(requests, ["12A"], use_lock=True))

> **Key idea.** The event loop is one thread, so a blocking call (time.sleep, blocking I/O, heavy computation) freezes every task: use async libraries, await asyncio.to_thread for blocking calls and run_in_executor with a process pool for CPU work; debug mode reports slow callbacks. Calling a coroutine function without await does nothing (and a coroutine object is always true). The loop keeps only weak references to tasks: hold them in a set with add_done_callback(discard), and observe their exceptions. CancelledError is a BaseException: clean up in finally, re-raise it if caught, and never swallow it with bare except. Races still happen across await, so keep check and act together or use asyncio.Lock. gather raises the first error and leaves the rest running (return_exceptions=True collects them); TaskGroup cancels the rest and raises an ExceptionGroup.

### Exercise: Who is blocking whom?

Simulate a single-threaded event loop. Each task is `(name, steps)`: `("wait", t)` lets other tasks run while it waits `t` units; `("block", t)` holds the loop for `t` units, so **nothing else** happens meanwhile; `("offload", t)` hands the work to a thread, so it's like a wait (the loop stays free). Write `simulate(tasks)`: tasks take turns in the order given; at each turn a task runs its next step; blocks advance the clock immediately; a waiting task is skipped until its wait is over; when every unfinished task is waiting, the clock jumps to the earliest wake-up. A task finishes when it has no steps left and isn't waiting. Return a dict of finish times. Then `worst_delay(tasks)`: the largest difference between a task's finish time with these steps and with every `"block"` turned into `"offload"`.

In [ ]:
def simulate(tasks):
    ___

def worst_delay(tasks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_blocking():\n    \"blocking freezes everyone\"\n    assert simulate([(\"a\", [(\"wait\", 5)]), (\"b\", [(\"wait\", 1)])]) == {\"b\": 1, \"a\": 5}\n    assert simulate([(\"a\", [(\"block\", 5)]), (\"b\", [(\"wait\", 1)])]) == {\"a\": 5, \"b\": 6}\n    assert simulate([(\"a\", [(\"offload\", 5)]), (\"b\", [(\"wait\", 1)])]) == {\"b\": 1, \"a\": 5}\n    mixed = [(\"web\", [(\"wait\", 1), (\"wait\", 1)]), (\"report\", [(\"block\", 3), (\"wait\", 1)])]\n    assert simulate(mixed) == {\"web\": 4, \"report\": 4}\n    assert worst_delay(mixed) == 2 and worst_delay([(\"x\", [(\"wait\", 2)])]) == 0\n    assert simulate([]) == {}")

### Exercise: Keeping background tasks alive

Write `TaskRegistry` to hold strong references to background tasks: `track(task)` adds it to a set and registers done callbacks that remove it again and record any **exception** it raised (but not cancellations) in `errors` as `(name, exception type name)`, where `name` is the name given to `track(task, name)`; `pending()` is how many are still running; and `track` returns the task. The tests use `concurrent.futures.Future` objects as stand-ins for tasks (they have the same `add_done_callback`, `cancelled` and `exception` methods).

In [ ]:
class TaskRegistry:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_registry():\n    \"strong references and observed errors\"\n    from concurrent.futures import Future\n    reg = TaskRegistry()\n    a, b, c = Future(), Future(), Future()\n    assert reg.track(a, \"email\") is a\n    reg.track(b, \"upload\")\n    reg.track(c, \"cleanup\")\n    assert reg.pending() == 3\n    a.set_result(\"sent\")\n    b.set_exception(ConnectionError(\"down\"))\n    c.cancel()\n    assert reg.pending() == 0 and reg.errors == [(\"upload\", \"ConnectionError\")]\n    done = Future()\n    done.set_result(1)\n    reg.track(done, \"already\")\n    assert reg.pending() == 0")

### Exercise: What gather does with failures

Model `asyncio.gather`'s error handling. Each outcome is a result value or an exception instance, listed in the **order given** to gather, with `finish_order` giving the indices in the order they finished. Write `gather_result(outcomes, finish_order, return_exceptions=False)`: with `return_exceptions=True`, return the outcomes as a list in input order (exceptions included as values); otherwise, if any outcome is an exception, raise the one that **finished first**, else return the list. Then `still_running(outcomes, finish_order)`: when gather (without `return_exceptions`) raises, the indices (sorted) of tasks that hadn't finished yet and keep running unobserved; an empty list if it doesn't raise.

In [ ]:
def gather_result(outcomes, finish_order, return_exceptions=False):
    ___

def still_running(outcomes, finish_order):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gather():\n    \"gather and errors\"\n    outcomes = [\"a\", ValueError(\"v\"), \"c\", KeyError(\"k\")]\n    order = [2, 3, 0, 1]\n    assert gather_result(outcomes, order, return_exceptions=True)[0] == \"a\"\n    assert isinstance(gather_result(outcomes, order, True)[3], KeyError)\n    try:\n        gather_result(outcomes, order)\n        assert False\n    except KeyError:\n        pass\n    except ValueError:\n        assert False, \"the first to finish was the KeyError\"\n    assert still_running(outcomes, order) == [0, 1]\n    assert gather_result([\"x\", \"y\"], [1, 0]) == [\"x\", \"y\"] and still_running([\"x\"], [0]) == []")

### Exercise: Does it respect cancellation?

Write `respects_cancellation(coro_fn)`: start a coroutine from `coro_fn()` by sending `None` once, throw `asyncio.CancelledError()` into it, and drive it until it ends (sending `None` whenever it suspends, at most 100 times). Return `True` if the `CancelledError` came back out, `False` if the coroutine kept running or finished normally instead. Close the coroutine when you're done. Then write `careful_worker(log)`, a coroutine that awaits the provided `Pause()` in an endless loop and appends `"closed"` to `log` in a `finally`, so it passes the check and cleans up.

In [ ]:
import asyncio

class Pause:
    def __await__(self):
        yield

def respects_cancellation(coro_fn):
    ___

async def careful_worker(log):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cancel():\n    \"CancelledError\"\n    import asyncio\n    log = []\n    assert respects_cancellation(lambda: careful_worker(log)) and log == [\"closed\"]\n    async def swallower():\n        while True:\n            try:\n                await Pause()\n            except asyncio.CancelledError:\n                pass\n    async def eats_then_returns():\n        try:\n            await Pause()\n        except asyncio.CancelledError:\n            return \"pretended nothing happened\"\n    async def catches_exception_only():\n        try:\n            while True:\n                await Pause()\n        except Exception:\n            return \"never reached\"\n    assert not respects_cancellation(swallower) and not respects_cancellation(eats_then_returns)\n    assert respects_cancellation(catches_exception_only)")

### Exercise: A race across an await

Two kinds of booking coroutine run under a round-robin driver: each booking checks whether a seat is free, then `await`s a save (letting the others run), then marks the seat taken. Write `book(seat, taken, booked, name)`: the racy version, appending `name` to `booked` if it found the seat free; and `book_locked(seat, taken, booked, name, lock)`: the same, inside `async with lock:` using the provided `SimLock`. Then `round_robin(coros)`: drive a list of coroutines one step each in turn until all are done (a coroutine suspended on a `SimLock` that's held is skipped until it's free: the lock tells you with `can_enter`). With the plain version two people book the same seat; with the lock, only one does.

In [ ]:
class Pause:
    def __await__(self):
        yield

class SimLock:
    "An asyncio-style lock for the round-robin driver."
    def __init__(self):
        self.owner = None
    def can_enter(self, coro):
        return self.owner is None or self.owner is coro
    async def __aenter__(self):
        while self.owner is not None and self.owner is not _current[0]:
            await Pause()
        self.owner = _current[0]
        return self
    async def __aexit__(self, *exc):
        self.owner = None
        return False

_current = [None]

async def book(seat, taken, booked, name):
    ___

async def book_locked(seat, taken, booked, name, lock):
    ___

def round_robin(coros):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_race():\n    \"check-then-act across await\"\n    taken, booked = set(), []\n    round_robin([book(\"12A\", taken, booked, \"ada\"), book(\"12A\", taken, booked, \"ben\")])\n    assert booked == [\"ada\", \"ben\"] and taken == {\"12A\"}\n    taken2, booked2, lock = set(), [], SimLock()\n    round_robin([book_locked(\"12A\", taken2, booked2, \"ada\", lock), book_locked(\"12A\", taken2, booked2, \"ben\", lock)])\n    assert booked2 == [\"ada\"] and lock.owner is None\n    taken3, booked3, lock3 = set(), [], SimLock()\n    round_robin([book_locked(s, taken3, booked3, n, lock3) for n, s in [(\"a\", \"1\"), (\"b\", \"2\"), (\"c\", \"1\")]])\n    assert booked3 == [\"a\", \"b\"]")

### Bug hunt: A coroutine that's always true

`is_banned` is a coroutine function, and the check calls it without `await`, so `if` tests a coroutine object, which is always true, and even Ada is refused: this prints `refused`. Await the check (inside an async function) so it prints `welcome`.

In [ ]:
async def is_banned(user):
    return user == "mallory"

async def enter(user):
    if is_banned(user):
        return "refused"
    return "welcome"

c = enter("ada")
try:
    c.send(None)
except StopIteration as done:
    print(done.value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"await the check\"\n    expect_output(\"welcome\")")

### Bug hunt: A cancellation that was eaten

The worker catches **everything** with `except BaseException`, including the `CancelledError` used to stop it, so cancelling it does nothing and this prints `still running`. Let the cancellation through (catch only `Exception`, or re-raise `CancelledError`) so it prints `cancelled`.

In [ ]:
import asyncio

class Pause:
    def __await__(self):
        yield

async def worker():
    while True:
        try:
            await Pause()
        except BaseException:
            pass

c = worker()
c.send(None)
try:
    c.throw(asyncio.CancelledError())
    print("still running")
except asyncio.CancelledError:
    print("cancelled")
try:
    c.close()
except RuntimeError:
    pass

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"don't swallow cancellation\"\n    expect_output(\"cancelled\")")

### Bug hunt: Tasks that are never forgotten

The background set keeps every task for ever, because nothing removes them when they finish, so it grows without limit and this prints `3`. Register `background.discard` as a done callback so finished tasks leave the set, and this prints `0`.

In [ ]:
from concurrent.futures import Future

background = set()

def spawn(task):
    background.add(task)
    return task

tasks = [spawn(Future()) for _ in range(3)]
for t in tasks:
    t.set_result("done")
print(len(background))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"discard when done\"\n    expect_output(\"0\")")

### Puzzle: Offloading a blocking call

Arrange the lines of a coroutine that reads a large file without blocking the event loop (real `asyncio`). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: issubclass(asyncio.CancelledError, Exception)
q = ___   # what time.sleep(2) inside a coroutine does to other tasks: "nothing" or "delays them"
r = ___   # bool of a coroutine object (one returning False)
s = ___   # what gather does with the other tasks when one raises: "cancels them" or "leaves them running"
t = ___   # what TaskGroup does with the other tasks when one raises: "cancels them" or "leaves them running"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == \"delays them\" and r is True, \"p, q and r\"\n    assert s == \"leaves them running\" and t == \"cancels them\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an async checker

Linters catch many async mistakes by reading the code (Week 64's `ast`). Write `check_async(source)`, returning a sorted list of `(line, code, message)` problems found **inside `async def` functions** (including nested ones):

| code | finds | message |
|---|---|---|
| `A001` | a call to `time.sleep(...)` | `blocking call time.sleep in async function` |
| `A002` | a call to `requests.get`, `requests.post`, `requests.put` or `requests.delete` | `blocking call requests.get in async function` (with the real name) |
| `A003` | a call, as a plain statement or a condition, to a coroutine function **defined in the same source** (any `async def` name), that isn't awaited | `coroutine 'name' is never awaited` |
| `A004` | `asyncio.create_task(...)` used as a plain statement (its result isn't kept) | `task created but no reference kept` |
| `A005` | an `except` that would swallow cancellation: a bare `except:`, `except BaseException`, or `except asyncio.CancelledError`, whose body doesn't end with a `raise` | `cancellation swallowed` |
| `A006` | `asyncio.run(...)` called inside an async function | `asyncio.run inside a running event loop` |

"Not awaited" means the `Call` node isn't the value of an `Await` node. A condition means the test of an `if` or `while`.

In [ ]:
import ast

def check_async(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checker():\n    \"finding async mistakes\"\n    src = (\n        \"import asyncio, time, requests\\n\"\n        \"async def save(x):\\n\"\n        \"    time.sleep(1)\\n\"\n        \"    return x\\n\"\n        \"async def main():\\n\"\n        \"    save(1)\\n\"\n        \"    await save(2)\\n\"\n        \"    if save(3):\\n\"\n        \"        pass\\n\"\n        \"    asyncio.create_task(save(4))\\n\"\n        \"    keep = asyncio.create_task(save(5))\\n\"\n        \"    r = requests.get('https://example.org')\\n\"\n        \"    try:\\n\"\n        \"        await save(6)\\n\"\n        \"    except asyncio.CancelledError:\\n\"\n        \"        print('ignored')\\n\"\n        \"    try:\\n\"\n        \"        await save(7)\\n\"\n        \"    except BaseException:\\n\"\n        \"        raise\\n\"\n        \"    asyncio.run(save(8))\\n\"\n        \"def sync_helper():\\n\"\n        \"    time.sleep(1)\\n\"\n    )\n    assert check_async(src) == [\n        (3, \"A001\", \"blocking call time.sleep in async function\"),\n        (6, \"A003\", \"coroutine 'save' is never awaited\"),\n        (8, \"A003\", \"coroutine 'save' is never awaited\"),\n        (10, \"A004\", \"task created but no reference kept\"),\n        (12, \"A002\", \"blocking call requests.get in async function\"),\n        (15, \"A005\", \"cancellation swallowed\"),\n        (21, \"A006\", \"asyncio.run inside a running event loop\"),\n    ]\n    assert check_async(\"async def f():\\n    try:\\n        pass\\n    except:\\n        pass\\n\") == [(4, \"A005\", \"cancellation swallowed\")]\n    assert check_async(\"def f():\\n    asyncio.run(g())\\n\") == []")

## Warm-up

When task `a` waits politely, `b` finishes at 1. When `a` **blocks** for the same five units (with `time.sleep(5)`, or five seconds of computation), `b` can't even start its wait until `a` is done, and finishes at 6. One blocking call delays every other task.

## Blocking the loop

The event loop runs on **one thread**, switching tasks only at `await`. Anything that holds the thread without awaiting stops **all** tasks:

- **`time.sleep(n)`** instead of `await asyncio.sleep(n)`;
- **blocking I/O libraries**: `requests.get`, ordinary file reads of large files, database drivers that aren't async, `input()`;
- **CPU-heavy work**: parsing a huge JSON file, image processing, a long loop.

Symptoms: everything gets slow at once, timeouts fire for no clear reason, and a server stops answering while one request is handled. `asyncio`'s **debug mode** (`asyncio.run(main(), debug=True)`, or `PYTHONASYNCIODEBUG=1`) logs any callback that runs for more than 100 milliseconds.

Fixes: use async libraries for I/O (`aiohttp`, `httpx`, `asyncpg` and others), and push unavoidable blocking calls onto a thread with **`await asyncio.to_thread(func, *args)`** (3.9+), or CPU-bound work onto a process pool with `loop.run_in_executor(pool, func, *args)`. The loop keeps running while the thread or process works.

```
import asyncio, time

async def bad():
    time.sleep(2)                          # freezes the loop for 2 seconds
async def good():
    await asyncio.sleep(2)                 # other tasks run meanwhile
async def wrapped():
    await asyncio.to_thread(time.sleep, 2) # a blocking call moved to a thread
```

## Forgetting await

Calling a coroutine function without `await` creates a coroutine object and **does nothing else**. The work never happens, and there's often no error at all:

- `save(record)` instead of `await save(record)`: nothing is saved; Python may print `RuntimeWarning: coroutine 'save' was never awaited` when the object is garbage collected, but only as a warning;
- `if is_allowed(user):` with an async `is_allowed`: a coroutine object is **always true**, so everyone is allowed;
- `results = [fetch(u) for u in urls]` builds a list of coroutines, not results (use `await asyncio.gather(*...)`).

Type checkers (Week 33's mypy) catch many of these, because the result's type is `Coroutine`, not `bool` or `str`. Linters warn too.

## Lost tasks and lost exceptions

`asyncio.create_task(job())` schedules a task, but the event loop holds only a **weak** reference to it. If your code doesn't keep a reference either, the task can be garbage collected **before it finishes**, vanishing mid-way. The documented fix is to keep tasks in a set and remove each when it's done:

```
background = set()

def spawn(coro):
    task = asyncio.create_task(coro)
    background.add(task)
    task.add_done_callback(background.discard)
    return task
```

A related problem: if a task raises and **nobody awaits it** (or calls `task.exception()`), the exception goes nowhere useful; `asyncio` logs "Task exception was never retrieved" much later, when the task is garbage collected. Every task should be awaited somewhere, or have a done callback that checks for errors. `asyncio.TaskGroup` (next section) solves both problems by tying tasks to a block.

## Cancellation done right

Cancelling a task (`task.cancel()`, or a timeout) raises **`asyncio.CancelledError`** inside it at its current `await`. Since Python 3.8 it inherits from `BaseException`, not `Exception`, precisely so that `except Exception:` doesn't catch it by accident. The rules:

- **Clean up in `finally`** (or `async with`): close connections, release locks, delete temporary files. Clean-up code may `await`.
- **Don't swallow it.** If you catch `CancelledError` (or use a bare `except:` or `except BaseException:`), re-raise it. A task that swallows cancellation keeps running, and `wait_for` or `TaskGroup` may wait for it for ever.
- **Don't treat cancellation as an error** to log loudly: it's normal control flow.
- `asyncio.shield(aw)` protects an inner operation from being cancelled along with its caller, for things that must finish (like committing a transaction).

```
async def worker(conn):
    try:
        while True:
            job = await queue.get()
            await handle(conn, job)
    finally:
        await conn.close()              # runs on cancellation too
```

## Races across await, and failing groups

One thread doesn't mean no races. A task can't be interrupted **between** awaits, but anything can happen **during** an `await`. Check-then-act across an `await` is a race:

```
async def book(seat):
    if seat not in taken:              # check
        await save_to_database(seat)   # other tasks run here, and may book the same seat
        taken.add(seat)                # act
```

Keep the check and the act with no `await` between them, or protect the whole sequence with an **`asyncio.Lock`** (`async with lock:`).

**When one task in a group fails**:

- `await asyncio.gather(a(), b(), c())` raises the **first** exception, but the other tasks **keep running** in the background, unwatched. `gather(..., return_exceptions=True)` returns exceptions in the result list instead of raising.
- `async with asyncio.TaskGroup() as tg:` (3.11+) **cancels** the other tasks when one fails, waits for them, and raises an `ExceptionGroup` of the failures (Week 59's `except*` handles those). It's usually the better choice: no task outlives the block.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run an `asyncio` program with `asyncio.run(main(), debug=True)` and put a `time.sleep(0.5)` in one coroutine: the debug output names the slow callback. Tomorrow: choosing between threads, processes and `asyncio`, and combining them.